# LSTM Return Forecasting

Standalone notebook for SBER OHLC forward-return forecasting with PyTorch LSTM models.

The notebook copies only the data loading and feature engineering needed from `ml.ipynb`, then trains one LSTM per horizon: `1d`, `7d`, `14d`, and `30d`. Data is split chronologically into `60%` train, `20%` validation, and `20%` test.

In [36]:
import math
import os
import random
import time
import warnings

import numpy as np
import optuna
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
import torch
from plotly.subplots import make_subplots
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.preprocessing import StandardScaler
from torch import nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
import json
import tempfile
from pathlib import Path

import joblib
import mlflow

try:
    from IPython.display import display
except ImportError:
    def display(obj):
        print(obj)

warnings.filterwarnings("ignore", category=UserWarning)
optuna.logging.set_verbosity(optuna.logging.INFO)
pio.renderers.default = "colab"

RANDOM_STATE = 42
random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)

device = torch.device("cpu")
print("device:", device)

device: cpu


## Data Loading And Feature Processing

This section mirrors only the preprocessing pieces needed for LSTM: load historical daily data, split symbols into separate time series, drop very short histories, keep the most recent 1200 rows, and build OHLC/return/volatility features.

In [37]:
df = pd.read_csv("ML/gen/historical_data_1d.csv", sep=";")
df["begin"] = pd.to_datetime(df["begin"])

df = df.sort_values(["name", "begin"]).reset_index(drop=True)

df = df[~df["name"].isin(["MBNK", "SVCB"])].copy()

df = df.groupby("name").tail(1200).reset_index(drop=True)

df.head(5)

,begin,open,close,high,low,value,volume,name
0,2021-03-12,6.980,6.895,7.029,6.884,415221145.0,59803300,CBOM
1,2021-03-15,6.901,6.891,6.923,6.882,371072061.5,53753800,CBOM
2,2021-03-16,6.890,6.812,6.895,6.811,611870520.4,89503200,CBOM
3,2021-03-17,6.819,6.725,6.860,6.693,623393436.0,92504800,CBOM
4,2021-03-18,6.720,6.939,6.949,6.720,546612371.5,79522900,CBOM


In [38]:
def add_return_features(df):
    df = df.sort_values(["name", "begin"]).copy()

    g = df.groupby("name")

    df["log_return"] = g["close"].transform(
        lambda s: np.log(s / s.shift(1))
    )

    df["intraday_return"] = df["close"] / df["open"] - 1.0

    return df


def add_candle_features(df):
    df = df.copy()

    df["upper_wick"] = (
        df["high"] - df[["open", "close"]].max(axis=1)
    ) / df["close"]

    df["lower_wick"] = (
        df[["open", "close"]].min(axis=1) - df["low"]
    ) / df["close"]

    df["amplitude"] = (df["high"] - df["low"]) / df["close"]

    return df

def add_volume_features(df):
    g = df.groupby("name")

    df["log_volume"] = np.log1p(df["volume"])

    df["volume_change"] = g["volume"].pct_change()

    return df

def add_volatility_features(df):
    g = df.groupby("name")

    for vol_tf in [7, 30]:
        df[f"volatility_{vol_tf}"] = g["log_return"].transform(
            lambda s: s.rolling(vol_tf).std().shift(1)
        )

    return df

df = df.replace([np.inf, -np.inf], np.nan)
df = df.dropna().reset_index(drop=True)
df = add_return_features(df)
df = add_candle_features(df)
df = add_volume_features(df)
df = add_volatility_features(df)
df["name_id"] = df["name"].astype("category").cat.codes

feature_cols = [c for c in df.columns if c not in ["begin", "name"]]
df[feature_cols] = df[feature_cols].replace([np.inf, -np.inf], np.nan)
df[feature_cols] = df[feature_cols].fillna(method="ffill")
df[feature_cols] = df[feature_cols].fillna(0.0)

print("rows after feature cleanup:", len(df))
print("date range:", df["begin"].min(), "->", df["begin"].max())
print("feature columns:", feature_cols)
display(df.head())

rows after feature cleanup: 6000
date range: 2021-02-05 00:00:00 -> 2025-10-13 00:00:00
feature columns: ['open', 'close', 'high', 'low', 'value', 'volume', 'log_return', 'intraday_return', 'upper_wick', 'lower_wick', 'amplitude', 'log_volume', 'volume_change', 'volatility_7', 'volatility_30', 'name_id']


C:\Users\shind\AppData\Local\Temp\ipykernel_27004\3628881280.py:59: FutureWarning:

DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.



,begin,open,close,high,low,value,volume,name,log_return,intraday_return,upper_wick,lower_wick,amplitude,log_volume,volume_change,volatility_7,volatility_30,name_id
0,2021-03-12,6.980,6.895,7.029,6.884,415221145.0,59803300,CBOM,0.000000,-0.012178,0.007107,0.001595,0.021030,17.906571,0.000000,0.0,0.0,0
1,2021-03-15,6.901,6.891,6.923,6.882,371072061.5,53753800,CBOM,-0.000580,-0.001449,0.003193,0.001306,0.005950,17.799925,-0.101157,0.0,0.0,0
2,2021-03-16,6.890,6.812,6.895,6.811,611870520.4,89503200,CBOM,-0.011530,-0.011321,0.000734,0.000147,0.012331,18.309785,0.665058,0.0,0.0,0
3,2021-03-17,6.819,6.725,6.860,6.693,623393436.0,92504800,CBOM,-0.012854,-0.013785,0.006097,0.004758,0.024833,18.342771,0.033536,0.0,0.0,0
4,2021-03-18,6.720,6.939,6.949,6.720,546612371.5,79522900,CBOM,0.031326,0.032589,0.001441,0.000000,0.033002,18.191556,-0.140338,0.0,0.0,0


## Metrics And Sequence Builder

Each sample contains a rolling feature window ending at date `t`. The target is log return based on close price:

`log(ohlc[future_pos] / ohlc[end_pos])`.

The split is chronological `60/20/20`; scalers are fit only on train data.

In [39]:
def return_smape(y_true, y_pred, eps=1e-12):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    denom = np.abs(y_true) + np.abs(y_pred) + eps
    return np.mean(200.0 * np.abs(y_pred - y_true) / denom)


def add_return_metrics(name, y_true, y_pred, df_metrics):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    df_metrics.loc[name] = [
        mean_squared_error(y_true, y_pred),
        mean_absolute_error(y_true, y_pred),
        return_smape(y_true, y_pred),
    ]


def split_positions(n_samples, train_ratio=0.60, val_ratio=0.20):
    train_end = int(n_samples * train_ratio)
    val_end = int(n_samples * (train_ratio + val_ratio))
    if train_end <= 0 or val_end <= train_end or val_end >= n_samples:
        raise ValueError(f"Bad split for {n_samples} samples")
    return train_end, val_end


def transform_sequences(scaler, x):
    n_samples, lookback, n_features = x.shape
    x_scaled = scaler.transform(x.reshape(-1, n_features))
    return x_scaled.reshape(n_samples, lookback, n_features).astype(np.float32)


def build_lstm_mimo_data(
    df,
    horizons,
    lookback,
    feature_cols,
):
    df = df.sort_values(["name", "begin"]).reset_index(drop=True).copy()
    df["begin"] = pd.to_datetime(df["begin"])

    unique_dates = np.sort(df["begin"].unique())
    n_dates = len(unique_dates)

    train_idx = int(n_dates * 0.60)
    val_idx = int(n_dates * 0.80)

    train_end_date = pd.to_datetime(unique_dates[train_idx])
    val_end_date = pd.to_datetime(unique_dates[val_idx])

    X_all, Y_all, TS_all, Close_all, name_all = [], [], [], [], []
    max_h = max(horizons)

    for name, g in df.groupby("name"):
        g = g.sort_values("begin").reset_index(drop=True)

        features = g[feature_cols].astype(float).to_numpy()
        close = g["close"].astype(float).to_numpy()
        timestamps = g["begin"].to_numpy()

        n = len(g)

        for end_pos in range(lookback, n - max_h):
            start_pos = end_pos - lookback

            x_win = features[start_pos:end_pos]

            if x_win.shape[0] != lookback:
                continue

            y_vec = np.array(
                [np.log(close[end_pos + h] / close[end_pos]) for h in horizons],
                dtype=np.float32,
            )

            X_all.append(x_win)
            Y_all.append(y_vec)
            TS_all.append(timestamps[end_pos])
            Close_all.append(close[end_pos])
            name_all.append(name)

    X = np.asarray(X_all, dtype=np.float32)
    Y = np.asarray(Y_all, dtype=np.float32)
    TS = pd.to_datetime(np.asarray(TS_all))
    names = np.asarray(name_all)
    Close = np.asarray(Close_all, dtype=np.float32)

    train_mask = TS <= pd.to_datetime(train_end_date)
    val_mask = (TS > pd.to_datetime(train_end_date)) & (TS <= pd.to_datetime(val_end_date))
    test_mask = TS > pd.to_datetime(val_end_date)

    masks = {
        "train": train_mask,
        "val": val_mask,
        "test": test_mask
    }

    feature_scaler = StandardScaler()
    X_train_flat = X[masks["train"]].reshape(-1, X.shape[-1])

    feature_scaler.fit(X_train_flat)

    def scale_x(x):
        n, t, f = x.shape
        return feature_scaler.transform(x.reshape(-1, f)).reshape(n, t, f).astype(np.float32)

    target_scalers = {}

    result = {
        "horizons": horizons,
        "lookback": lookback,
        "feature_cols": feature_cols,
        "n_features": X.shape[-1],
        "feature_scaler": feature_scaler,
        "target_scalers": target_scalers,
    }

    for name, mask in masks.items():
        result[f"X_{name}"] = scale_x(X[mask])
        result[f"y_{name}"] = Y[mask]
        result[f"y_{name}_return"] = Y[mask]
        result[f"current_close_{name}"] = Close[mask]
        result[f"ts_{name}"] = TS[mask]
        result[f"name_{name}"] = names[mask]

    return result

## LSTM Model

The nn.Linear(hidden_size, output_size) layer is the regression head.
It maps the final hidden representation of the sequence into multiple continuous forecasts (multi-horizon return prediction).

The LSTM encodes the input sequence into a latent representation,
and the head projects it into horizon-specific return forecasts.

The model is trained end-to-end.

In [40]:
def huber_directional_loss(pred, target, delta=1.0, alpha=0.5):
    loss_huber = F.huber_loss(pred, target, delta=delta, reduction='mean')

    pred_sign = torch.tanh(pred * 50.0)
    target_sign = torch.tanh(target * 50.0)

    sign_product = pred_sign * target_sign

    loss_direction = F.relu(-sign_product).mean()

    return loss_huber + alpha * loss_direction


class ReturnLSTM(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, dropout, output_size=4):
        super().__init__()

        lstm_dropout = dropout if num_layers > 1 else 0.0

        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            dropout=lstm_dropout,
            batch_first=True,
        )

        self.head = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        output, _ = self.lstm(x)
        last_hidden = output[:, -1, :]
        return self.head(last_hidden)


def make_loader(x, y, batch_size, shuffle=False):
    dataset = TensorDataset(
        torch.as_tensor(x, dtype=torch.float32),
        torch.as_tensor(y, dtype=torch.float32),
    )
    return DataLoader(dataset, batch_size=batch_size, shuffle=shuffle)


def evaluate_loss(model, loader, delta=1.0, alpha=0.5):
    model.eval()
    total_loss = 0.0
    total_count = 0
    with torch.no_grad():
        for x_batch, y_batch in loader:
            x_batch = x_batch.to(device)
            y_batch = y_batch.to(device)
            pred = model(x_batch)
            loss = huber_directional_loss(pred, y_batch, delta=delta, alpha=alpha)

            total_loss += loss.item() * len(y_batch)
            total_count += len(y_batch)
    return total_loss / max(total_count, 1)


def train_lstm_model(
    data,
    params,
    max_epochs=100,
    patience=12,
    verbose=False,
):
    torch.manual_seed(RANDOM_STATE)
    model = ReturnLSTM(
        input_size=data["n_features"],
        hidden_size=int(params["hidden_size"]),
        num_layers=int(params["num_layers"]),
        dropout=float(params["dropout"]),
    ).to(device)

    batch_size = int(params["batch_size"])
    train_loader = make_loader(data["X_train"], data["y_train"], batch_size=batch_size, shuffle=False)
    val_loader = make_loader(data["X_val"], data["y_val"], batch_size=batch_size, shuffle=False)

    alpha_weight = float(params.get("direction_alpha", 0.5))

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=float(params["learning_rate"]),
        weight_decay=float(params["weight_decay"]),
    )

    best_val_loss = np.inf
    best_epoch = 0
    best_state = None
    no_improve = 0
    train_losses = []
    val_losses = []

    for epoch in range(1, max_epochs + 1):
        model.train()
        epoch_loss = 0.0
        epoch_count = 0

        for x_batch, y_batch in train_loader:
            x_batch = x_batch.to(device)
            y_batch = y_batch.to(device)

            optimizer.zero_grad(set_to_none=True)
            pred = model(x_batch)

            loss = huber_directional_loss(pred, y_batch, delta=1.0, alpha=alpha_weight)

            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()

            epoch_loss += loss.item() * len(y_batch)
            epoch_count += len(y_batch)

        train_loss = epoch_loss / max(epoch_count, 1)

        val_loss = evaluate_loss(model, val_loader, delta=1.0, alpha=alpha_weight)

        train_losses.append(train_loss)
        val_losses.append(val_loss)

        if val_loss < best_val_loss - 1e-8:
            best_val_loss = val_loss
            best_epoch = epoch
            best_state = {key: value.detach().cpu().clone() for key, value in model.state_dict().items()}
            no_improve = 0
        else:
            no_improve += 1

        if verbose and (epoch == 1 or epoch % 10 == 0):
            print(f"epoch={epoch:03d} train_loss={train_loss:.6f} val_loss={val_loss:.6f}")

        if no_improve >= patience:
            break

    if best_state is not None:
        model.load_state_dict(best_state)

    history = {
        "train_loss": train_losses,
        "val_loss": val_losses,
        "best_val_loss": float(best_val_loss),
        "best_epoch": int(best_epoch),
        "epochs_ran": len(train_losses),
    }
    return model, history


def predict_mimo_returns(model, data, split_name="test", batch_size=64):
    x = data[f"X_{split_name}"]

    H = len(data["horizons"])
    y_dummy = np.zeros((len(x), H), dtype=np.float32)

    loader = make_loader(x, y_dummy, batch_size=batch_size, shuffle=False)

    model.eval()
    preds = []

    with torch.no_grad():
        for x_batch, _ in loader:
            preds.append(model(x_batch.to(device)).cpu().numpy())

    return np.concatenate(preds)


def returns_to_prices(current_close, predicted_returns):
    return current_close[:, None] * np.exp(predicted_returns)

def add_mimo_metrics(horizon_config, y_true, y_pred, df_metrics, name_prefix="lstm"):
    for i, h in enumerate(horizon_config):

        yt = y_true[:, i]
        yp = y_pred[:, i]

        mse = mean_squared_error(yt, yp)
        mae = mean_absolute_error(yt, yp)
        smape = return_smape(yt, yp)

        df_metrics.loc[f"{name_prefix}_{h}d"] = [mse, mae, smape]

def add_zero_baseline_mimo(horizon_config, y_true, df_metrics):
    zero_pred = np.zeros_like(y_true)

    for i, h in enumerate(horizon_config):

        yt = y_true[:, i]
        yp = zero_pred[:, i]

        mse = mean_squared_error(yt, yp)
        mae = mean_absolute_error(yt, yp)
        smape = return_smape(yt, yp)

        df_metrics.loc[f"zero_baseline_{h}d"] = [mse, mae, smape]

## MLFlow log functions

In [41]:
def safe_metric_name(name):
    return str(name).replace(" ", "_").replace("/", "_").replace(":", "_")


def log_dataframe_csv(df, artifact_path, filename):
    with tempfile.TemporaryDirectory() as tmpdir:
        path = Path(tmpdir) / filename
        df.to_csv(path, index=False)
        mlflow.log_artifact(str(path), artifact_path=artifact_path)


def log_json_dict(obj, artifact_path, filename):
    with tempfile.TemporaryDirectory() as tmpdir:
        path = Path(tmpdir) / filename
        with open(path, "w", encoding="utf-8") as f:
            json.dump(obj, f, ensure_ascii=False, indent=2, default=str)
        mlflow.log_artifact(str(path), artifact_path=artifact_path)


def log_plotly_figure(fig, artifact_path, filename_base):
    with tempfile.TemporaryDirectory() as tmpdir:
        path = Path(tmpdir) / f"{filename_base}.html"
        fig.write_html(str(path))
        mlflow.log_artifact(str(path), artifact_path=artifact_path)


def log_sklearn_object(obj, artifact_path, filename):
    with tempfile.TemporaryDirectory() as tmpdir:
        path = Path(tmpdir) / filename
        joblib.dump(obj, path)
        mlflow.log_artifact(str(path), artifact_path=artifact_path)


def make_predictions_df(data, y_pred, split="test", horizons=None):
    ts = data[f"ts_{split}"]
    y_true = data[f"y_{split}"]

    df = pd.DataFrame({
        "timestamp": ts.values,
    })

    n_horizons = y_pred.shape[1]

    for i in range(n_horizons):
        h_name = horizons[i] if horizons is not None else i

        df[f"y_true_h{h_name}"] = y_true[:, i]
        df[f"y_pred_h{h_name}"] = y_pred[:, i]

    return df


def log_optuna_trials(study, artifact_path="optuna"):
    trials_df = study.trials_dataframe()
    if len(trials_df) > 0:
        log_dataframe_csv(trials_df, artifact_path, "trials.csv")

## Optuna Search And Final Training

In [42]:
mlflow.set_tracking_uri("sqlite:///mlflow.db")
mlflow.set_experiment("stock_forecasting_experiment")

def make_mimo_objective(df, horizons, feature_cols):

    def objective(trial):

        params = {
            "lookback": trial.suggest_categorical("lookback", [30, 60, 90]),
            "hidden_size": trial.suggest_categorical("hidden_size", [32, 64, 128]),
            "num_layers": trial.suggest_int("num_layers", 1, 2),
            "dropout": trial.suggest_float("dropout", 0.0, 0.35),
            "learning_rate": trial.suggest_float("learning_rate", 1e-4, 3e-3, log=True),
            "weight_decay": trial.suggest_float("weight_decay", 1e-6, 1e-3, log=True),
            "batch_size": trial.suggest_categorical("batch_size", [16, 32, 64]),
            "direction_alpha": trial.suggest_float("direction_alpha", 0.3, 0.5),
        }

        data = build_lstm_mimo_data(
            df=df,
            horizons=horizons,
            lookback=params["lookback"],
            feature_cols=feature_cols,
        )

        model, history = train_lstm_model(
            data=data,
            params=params,
            max_epochs=80,
            patience=10,
            verbose=False,
        )

        trial.set_user_attr("best_epoch", history["best_epoch"])
        trial.set_user_attr("epochs_ran", history["epochs_ran"])

        return history["best_val_loss"]

    return objective

In [ ]:
horizon_config = [1, 7, 14, 30]
n_trials_lstm = 2
run_id = f"mimo_{'_'.join(map(str, horizon_config))}"

output_size = len(horizon_config)

lstm_mimo_datasets = {}
lstm_mimo_models = {}
lstm_mimo_predictions = {}
lstm_mimo_price_predictions = {}
lstm_mimo_best_params = {}
lstm_mimo_histories = {}

df_metrics_lstm_mimo = pd.DataFrame(columns=["MSE", "MAE", "sMAPE"])

timing_start = time.perf_counter()

with mlflow.start_run(run_name=f"lstm_mimo_all_horizons_trials_{n_trials_lstm}") as parent_run:
    mlflow.log_param("horizons", str(horizon_config))
    mlflow.log_param("n_trials_lstm", n_trials_lstm)
    mlflow.log_param("random_state", RANDOM_STATE)
    mlflow.log_param("device", str(device))
    mlflow.log_param("train_ratio", 0.60)
    mlflow.log_param("val_ratio", 0.20)
    mlflow.log_param("n_base_rows", len(df))
    mlflow.log_param("n_features", len(feature_cols))
    mlflow.log_param("feature_cols", ",".join(feature_cols))

    log_json_dict(
        {
            "feature_cols": feature_cols,
            "horizons": horizon_config,
            "random_state": RANDOM_STATE,
        },
        artifact_path="config",
        filename="run_config.json",
    )

    log_dataframe_csv(
        df.head(100),
        artifact_path="data",
        filename="df_head.csv",
    )

    study = optuna.create_study(
        direction="minimize",
        study_name="lstm_mimo",
        sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE),
    )

    optuna_start = time.perf_counter()

    study.optimize(
        make_mimo_objective(df, horizon_config, feature_cols),
        n_trials=n_trials_lstm,
        show_progress_bar=False,
    )

    optuna_seconds = time.perf_counter() - optuna_start

    best_params = dict(study.best_params)

    mlflow.log_metric("optuna_best_val_mse_scaled", float(study.best_value))
    mlflow.log_metric("optuna_seconds", float(optuna_seconds))
    mlflow.log_params({f"best_{k}": v for k, v in best_params.items()})

    log_optuna_trials(study, artifact_path="optuna")

    data = build_lstm_mimo_data(
        df=df,
        horizons=horizon_config,
        lookback=best_params["lookback"],
        feature_cols=feature_cols,
    )
    lstm_mimo_datasets[run_id] = data

    mlflow.log_param("lookback", best_params["lookback"])

    model, history = train_lstm_model(
        data=data,
        params=best_params,
        max_epochs=120,
        patience=15,
        verbose=False,
    )

    final_train_seconds = time.perf_counter() - optuna_start

    y_pred = predict_mimo_returns(
        model=model,
        data=data,
        split_name="test",
        batch_size=best_params["batch_size"],
    )

    y_test = data["y_test_return"]
    ts_test = data["ts_test"]

    current_close = data["current_close_test"]

    add_zero_baseline_mimo(
        horizon_config,
        y_test,
        df_metrics_lstm_mimo
    )

    add_mimo_metrics(
        horizon_config,
        y_test,
        y_pred,
        df_metrics_lstm_mimo,
        name_prefix="lstm"
    )

    price_pred = returns_to_prices(
        current_close,
        y_pred,
    )

    price_true = returns_to_prices(
        current_close,
        y_test,
    )

    mse_per_horizon = np.mean(
        (y_test - y_pred) ** 2,
        axis=0
    )

    mae_per_horizon = np.mean(
        np.abs(y_test - y_pred),
        axis=0
    )

    overall_mse = float(np.mean(mse_per_horizon))
    overall_mae = float(np.mean(mae_per_horizon))
    overall_smape = return_smape(y_test, y_pred)


    mlflow.log_metric(
        "test_return_mse",
        overall_mse
    )

    mlflow.log_metric(
        "test_return_mae",
        overall_mae
    )

    mlflow.log_metric(
        "test_return_smape",
        overall_smape
    )

    price_mse_per_horizon = np.mean(
        (price_true - price_pred) ** 2,
        axis=0
    )

    price_mae_per_horizon = np.mean(
        np.abs(price_true - price_pred),
        axis=0
    )

    price_rmse_per_horizon = np.sqrt(
        price_mse_per_horizon
    )

    overall_price_mse = float(
        np.mean(price_mse_per_horizon)
    )

    overall_price_mae = float(
        np.mean(price_mae_per_horizon)
    )

    overall_price_rmse = float(
        np.mean(price_rmse_per_horizon)
    )


    mlflow.log_metric(
        "test_price_mse",
        overall_price_mse
    )

    mlflow.log_metric(
        "test_price_mae",
        overall_price_mae
    )

    mlflow.log_metric(
        "test_price_rmse",
        overall_price_rmse
    )

    for i, h in enumerate(horizon_config):
        mlflow.log_metric(
            f"return_mse_h_{h}",
            float(mse_per_horizon[i])
        )

        mlflow.log_metric(
            f"return_mae_h_{h}",
            float(mae_per_horizon[i])
        )

        mlflow.log_metric(
            f"price_mse_h_{h}",
            float(price_mse_per_horizon[i])
        )

        mlflow.log_metric(
            f"price_mae_h_{h}",
            float(price_mae_per_horizon[i])
        )

        mlflow.log_metric(
            f"price_rmse_h_{h}",
            float(price_rmse_per_horizon[i])
        )


    lstm_mimo_models[run_id] = model
    lstm_mimo_predictions[run_id] = y_pred
    lstm_mimo_price_predictions[run_id] = price_pred
    lstm_mimo_histories[run_id] = history

    lstm_mimo_best_params = {
        **best_params,
        "best_val_mse_scaled": study.best_value,
        "best_epoch": history["best_epoch"],
    }

    history_df = pd.DataFrame({
        "epoch": np.arange(len(history["train_loss"])),
        "train_loss": history["train_loss"],
        "val_loss": history["val_loss"],
    })

    log_dataframe_csv(history_df, "history", "history.csv")

    return_predictions_df = pd.DataFrame(
        y_pred,
        columns=[f"t+{h}" for h in horizon_config],
        index=ts_test,
    )

    price_predictions_df = pd.DataFrame(
        price_pred,
        columns=[f"t+{h}" for h in horizon_config],
        index=ts_test,
    )

    price_actual_df = pd.DataFrame(
        price_true,
        columns=[f"t+{h}" for h in horizon_config],
        index=ts_test,
    )


    log_dataframe_csv(price_predictions_df, "predictions", "price_predictions.csv")
    log_dataframe_csv(price_actual_df, "predictions", "price_actual.csv")
    log_dataframe_csv(return_predictions_df, "predictions", "return_predictions.csv")

    log_sklearn_object(data["feature_scaler"], "scalers", "feature_scaler.joblib")
    log_sklearn_object(data["target_scalers"], "scalers", "target_scalers.joblib")

    mlflow.pytorch.log_model(
        model,
        artifact_path="model_mimo",
        input_example=data["X_test"][:1],
        registered_model_name="lstm_mimo_returns"
    )

    total_seconds = time.perf_counter() - timing_start
    mlflow.log_metric("total_seconds", float(total_seconds))

[I 2026-06-13 23:53:45,392] A new study created in memory with name: lstm_mimo
[I 2026-06-13 23:55:10,694] Trial 0 finished with value: 0.03278137929737568 and parameters: {'lookback': 60, 'hidden_size': 32, 'num_layers': 1, 'dropout': 0.3031616510212273, 'learning_rate': 0.0007725378389307352, 'weight_decay': 0.000133112160807369, 'batch_size': 32, 'direction_alpha': 0.34246782213565524}. Best is trial 0 with value: 0.03278137929737568.


In [ ]:
display(df_metrics_lstm_mimo)

## Return Forecast Plots

In [ ]:
tickers_to_show = df["name"].unique()

data = lstm_mimo_datasets[run_id]
preds = lstm_mimo_predictions[run_id]

for t_idx, ticker in enumerate(tickers_to_show):

    fig_return = make_subplots(
        rows=2,
        cols=2,
        subplot_titles=[f"{h}d return" for h in horizon_config],
        horizontal_spacing=0.08,
        vertical_spacing=0.12,
    )

    for h_idx, h in enumerate(horizon_config):
        mask = (data["name_test"] == ticker)

        ts = data["ts_test"][mask]

        fig_return.add_trace(
            go.Scatter(
                x=ts,
                y=data["y_test_return"][mask, h_idx],
                name=f"{ticker}|{h} actual",
                line=dict(dash="dot"),
            ),
            row=1 + h_idx // 2,
            col=1 + h_idx % 2,
        )

        fig_return.add_trace(
            go.Scatter(
                x=ts,
                y=preds[mask, h_idx],
                name=f"{ticker}|{h} pred",
            ),
            row=1 + h_idx // 2,
            col=1 + h_idx % 2,
        )

    fig_return.update_layout(
        width=1400,
        height=750,
        title="MIMO LSTM: forward returns forecast",
    )

    fig_return.show()

## Linear-Style Price Plots

Predicted returns are converted back to price level with:

`predicted_ohlc = current_ohlc * (1 + predicted_return)`.

In [ ]:
data = lstm_mimo_datasets[run_id]
preds = lstm_mimo_price_predictions[run_id]

for ticker in tickers_to_show:
    fig_price = make_subplots(
        rows=2,
        cols=2,
        subplot_titles=[f"{h}d price" for h in horizon_config],
        horizontal_spacing=0.08,
        vertical_spacing=0.12,
    )

    mask_ticker = (data["name_test"] == ticker)

    ts = data["ts_test"][mask_ticker]

    for h_idx, h in enumerate(horizon_config):

        row = (h_idx // 2) + 1
        col = (h_idx % 2) + 1

        y_true = data["current_close_test"][mask_ticker]
        y_pred = preds[mask_ticker, h_idx]

        fig_price.add_trace(
            go.Scatter(
                x=ts,
                y=y_true,
                mode="lines",
                name=f"{ticker}|{h} actual",
                line=dict(dash="dot"),
            ),
            row=row,
            col=col,
        )

        fig_price.add_trace(
            go.Scatter(
                x=ts,
                y=y_pred,
                mode="lines",
                name=f"{ticker}|{h} pred",
            ),
            row=row,
            col=col,
        )

    fig_price.update_layout(
        width=1400,
        height=750,
        title=f"MIMO LSTM: price forecast ({ticker})",
    )

    fig_price.show()

## Training Curves

In [ ]:
history = lstm_mimo_histories[run_id]

epochs = np.arange(1, len(history["train_loss"]) + 1)

fig_loss = make_subplots(
    rows=1,
    cols=1,
    subplot_titles=["MIMO LSTM training loss"],
)

fig_loss.add_trace(
    go.Scatter(
        x=epochs,
        y=history["train_loss"],
        mode="lines",
        name="train_loss",
        line=dict(color="blue"),
    )
)

fig_loss.add_trace(
    go.Scatter(
        x=epochs,
        y=history["val_loss"],
        mode="lines",
        name="val_loss",
        line=dict(color="orange"),
    )
)

fig_loss.update_layout(
    width=1000,
    height=500,
    title="MIMO LSTM training curves",
)

fig_loss.show()